In [1]:
import requests
KEY = "cf43da137142625295781d3c1db3a5bd"
r = requests.get("https://api.themoviedb.org/3/movie/550", params={"api_key": KEY})
print(r.status_code, r.text)

200 {"adult":false,"backdrop_path":"/c6OLXfKAk5BKeR6broC8pYiCquX.jpg","belongs_to_collection":null,"budget":63000000,"genres":[{"id":18,"name":"Drama"},{"id":53,"name":"Thriller"}],"homepage":"https://www.20thcenturystudios.com/movies/fight-club","id":550,"imdb_id":"tt0137523","origin_country":["US"],"original_language":"en","original_title":"Fight Club","overview":"A ticking-time-bomb insomniac and a slippery soap salesman channel primal male aggression into a shocking new form of therapy. Their concept catches on, with underground \"fight clubs\" forming in every town, until an eccentric gets in the way and ignites an out-of-control spiral toward oblivion.","popularity":43.0644,"poster_path":"/jSziioSwPVrOy9Yow3XhWIBDjq1.jpg","production_companies":[{"id":711,"logo_path":"/tEiIH5QesdheJmDAqQwvtN60727.png","name":"Fox 2000 Pictures","origin_country":"US"},{"id":508,"logo_path":"/4sGWXoboEkWPphI6es6rTmqkCBh.png","name":"Regency Enterprises","origin_country":"US"},{"id":4700,"logo_path"

In [11]:
import requests, json

KEY  = "cf43da137142625295781d3c1db3a5bd"
BASE = "https://api.themoviedb.org/3"

genres = {g["id"]: g["name"] for g in
          requests.get(f"{BASE}/genre/movie/list", params={"api_key": KEY}).json()["genres"]}

movies = []
page = 1
while len(movies) < 10000:
    r = requests.get(f"{BASE}/discover/movie", params={
        "api_key": KEY,
        "sort_by": "popularity.desc",
        "include_adult": "false",
        "vote_count.gte": 50,        # skips obscure entries
        "page": page})
    data = r.json()
    for m in data["results"]:
        if m.get("overview"):        # skip movies with no description
            movies.append({
                "movie_name": m["title"],
                "description": m["overview"],
                "genre": [genres.get(g) for g in m["genre_ids"]]
            })
    if page >= data["total_pages"]:
        break
    page += 1

movies = movies[:10000]
with open("movies.json", "w", encoding="utf-8") as f:
    json.dump(movies, f, ensure_ascii=False, indent=2)

print(len(movies), "movies saved")

10000 movies saved


In [35]:
import pandas as pd
df = pd.DataFrame(movies)
display(df.head())

,movie_name,description,genre
0,Spider-Man: Brand New Day,Fighting crime full-time as Spider-Man in a wo...,"[Science Fiction, Action, Adventure]"
1,Project Hail Mary,Science teacher Ryland Grace wakes up on a spa...,"[Science Fiction, Adventure]"
2,Hope,In the remote South Korean village of Hope Har...,"[Science Fiction, Mystery, Action]"
3,The Odyssey,"Odysseus, the legendary King of Ithaca, embark...","[Adventure, Action, Fantasy]"
4,Clayface,Matt Hagen's meteoric rise from Gotham street ...,"[Horror, Thriller, Science Fiction]"


In [36]:
df.shape

(10000, 3)

# **1.Lower**

In [37]:
df = df.map(lambda x: x.lower() if isinstance(x,str) else x)
# Convert all elements inside the list to lowercase
df['genre'] = df['genre'].apply(lambda x: [item.lower() for item in x] if isinstance(x, list) else x)


In [38]:
df.head()

,movie_name,description,genre
0,spider-man: brand new day,fighting crime full-time as spider-man in a wo...,"[science fiction, action, adventure]"
1,project hail mary,science teacher ryland grace wakes up on a spa...,"[science fiction, adventure]"
2,hope,in the remote south korean village of hope har...,"[science fiction, mystery, action]"
3,the odyssey,"odysseus, the legendary king of ithaca, embark...","[adventure, action, fantasy]"
4,clayface,matt hagen's meteoric rise from gotham street ...,"[horror, thriller, science fiction]"


In [39]:
print(type(df['genre'].iloc[0]))

<class 'list'>


In [41]:
df['genre'] = df['genre'].str.join(', ')

In [42]:
df

,movie_name,description,genre
0,spider-man: brand new day,fighting crime full-time as spider-man in a wo...,"science fiction, action, adventure"
1,project hail mary,science teacher ryland grace wakes up on a spa...,"science fiction, adventure"
2,hope,in the remote south korean village of hope har...,"science fiction, mystery, action"
3,the odyssey,"odysseus, the legendary king of ithaca, embark...","adventure, action, fantasy"
4,clayface,matt hagen's meteoric rise from gotham street ...,"horror, thriller, science fiction"
...,...,...,...
9995,major league,when rachel phelps inherits the cleveland indi...,comedy
9996,lupin the third vs. detective conan,when a wealthy woman and her son die in a hunt...,"tv movie, animation, action, comedy, crime"
9997,absolutely fabulous: the movie,edina and patsy are still oozing glitz and gla...,comedy
9998,don't look at the demon,a spiritual medium leads a paranormal investig...,horror


# 2.**REGEX**

In [46]:
# 2.now remove the regex from the all columns if available in it

import string

punc = f"[{string.punctuation}]"

df = df.apply(lambda X: X.str.replace(punc,"",regex=True) if X.dtype == "object" else X)

In [47]:
df['movie_name'][9996]

'lupin the third vs detective conan'

# **3.Tokenization:**

Tokenization is always used first befor Stemming or Lemmitization cause we have to iterate all word seperatly .
There are 2 technique of doing Tokenization : 1.WordbyWord , 2.SentancebySentance  here we do 1st

In [52]:
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize

# 1. Download the required punctuation resource (run once)
nltk.download('punkt')
nltk.download('punkt_tab')

# 2. Load your whole dataset

# 3. Apply tokenization to the entire text column
# Replace 'text_column' with the actual name of your column
df['tokens'] = df['description'].astype(str).apply(word_tokenize)

# 4. Save your newly tokenized dataset
df.to_csv("tokenized_dataset.csv", index=False)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [59]:
print(df[['description', 'tokens']].head())

                                         description  \
0  fighting crime fulltime as spiderman in a worl...   
1  science teacher ryland grace wakes up on a spa...   
2  in the remote south korean village of hope har...   
3  odysseus the legendary king of ithaca embarks ...   
4  matt hagens meteoric rise from gotham street k...   

                                              tokens  
0  [fighting, crime, fulltime, as, spiderman, in,...  
1  [science, teacher, ryland, grace, wakes, up, o...  
2  [in, the, remote, south, korean, village, of, ...  
3  [odysseus, the, legendary, king, of, ithaca, e...  
4  [matt, hagens, meteoric, rise, from, gotham, s...  


# **4.Stemming**

stemming is a technique which is used to remove the suffix and prefix from the word and convert that into a base form but there is not a gurranty that word is exist in dictonaries so it could be meaning less . There is two type of Stemming
1.PORTERSTEMMER
2.SNOWBALLSTEMMER
where 2nd is multilanguange and adv version of 1.

In [62]:
from nltk.stem import PorterStemmer,SnowballStemmer

porter= PorterStemmer()
snowball = SnowballStemmer('english')
# Stemming kayru pan As we can see that ene badha mathi e kadhi nakhyu je right nathi
df["porter_stem"] = df["tokens"].apply(lambda token_list : [porter.stem(word ) for word in token_list])
df["snowball_stem"] = df["tokens"].apply(lambda token_list : [snowball.stem(word ) for word in token_list])


# aaya join etle j kayru chhe ke e normal stem token ne jodi ne normal sentance return kare
df['PorterStemming'] = df["porter_stem"].apply(lambda x:''.join(x))
df['SnowballStemming'] = df["snowball_stem"].apply(lambda x:''.join(x))

In [63]:
df["PorterStemming"]

,PorterStemming
0,fightcrimefulltimasspidermaninaworldthatdoesnt...
1,sciencteacherrylandgracewakeuponaspaceshipligh...
2,intheremotsouthkoreanvillagofhopeharborpolicch...
3,odysseuthelegendarikingofithacaembarkonalongan...
4,matthagenmeteorrisefromgothamstreetkidtohollyw...
...,...
9995,whenrachelphelpinherittheclevelandindianfromhe...
9996,whenawealthiwomanandhersondieinahuntacciditupt...
9997,edinaandpatsiarestilloozglitzandglamourlivethe...
9998,aspiritumediumleadaparanorminvestigtvcrewtoaha...


In [64]:
df["snowball_stem"]

,snowball_stem
0,"[fight, crime, fulltim, as, spiderman, in, a, ..."
1,"[scienc, teacher, ryland, grace, wake, up, on,..."
2,"[in, the, remot, south, korean, villag, of, ho..."
3,"[odysseus, the, legendari, king, of, ithaca, e..."
4,"[matt, hagen, meteor, rise, from, gotham, stre..."
...,...
9995,"[when, rachel, phelp, inherit, the, cleveland,..."
9996,"[when, a, wealthi, woman, and, her, son, die, ..."
9997,"[edina, and, patsi, are, still, ooz, glitz, an..."
9998,"[a, spiritu, medium, lead, a, paranorm, invest..."


# **5.Lemitization**


In Lemitization its same as the Stemming but the main difference is that Lemitization convert word into 'meaningfull word' not just anything like Stemming.

In [70]:
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')
# nltk.download('omw-1.4')
lemmitizor = WordNetLemmatizer()

def lemitization(token_lists):
  if isinstance(token_lists,list):
    return [lemmitizor.lemmatize(word,'v') for word in token_lists]
  return token_lists

df["Lemitization"] = df["tokens"].apply(lemitization)

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [71]:
df["Lemitization"]

,Lemitization
0,"[fight, crime, fulltime, as, spiderman, in, a,..."
1,"[science, teacher, ryland, grace, wake, up, on..."
2,"[in, the, remote, south, korean, village, of, ..."
3,"[odysseus, the, legendary, king, of, ithaca, e..."
4,"[matt, hagens, meteoric, rise, from, gotham, s..."
...,...
9995,"[when, rachel, phelps, inherit, the, cleveland..."
9996,"[when, a, wealthy, woman, and, her, son, die, ..."
9997,"[edina, and, patsy, be, still, ooze, glitz, an..."
9998,"[a, spiritual, medium, lead, a, paranormal, in..."


In [74]:
df['str_lemmitization'] = df['Lemitization'].apply(lambda x: " ".join(x))

In [75]:
df['str_lemmitization']

,str_lemmitization
0,fight crime fulltime as spiderman in a world t...
1,science teacher ryland grace wake up on a spac...
2,in the remote south korean village of hope har...
3,odysseus the legendary king of ithaca embark o...
4,matt hagens meteoric rise from gotham street k...
...,...
9995,when rachel phelps inherit the cleveland india...
9996,when a wealthy woman and her son die in a hunt...
9997,edina and patsy be still ooze glitz and glamou...
9998,a spiritual medium lead a paranormal investiga...


In [76]:
df.to_csv("Lemmitized_data.csv", index=False)